# Assignment: Preparing Data for Analysis (Modified Titanic)

**Note:** Each problem starts from the original input DataFrame and is evaluated independently. Do not carry over results or modifications from previous problems. Repeat a previous step only when explicitly instructed.


![](https://github.com/kaopanboonyuen/2110446_DataScience_2021s2/raw/main/%20files/hw.png)

# 1) Load data & review the data

How many rows are there in the "titanic_to_student.csv"?

In [1]:
# Import the libraries
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [119]:
df = pd.read_csv('./titanic_to_student.csv', index_col=0)

In [3]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,2,1.0,1.0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0,PC 17599,71.2833,C85,C
1,4,1.0,1.0,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0,113803,53.1000,C123,S
2,6,0.0,3.0,"Moran, Mr. James",male,NaN,0.0,0,330877,8.4583,NaN,Q
3,8,0.0,3.0,"Palsson, Master. Gosta Leonard",male,2.0,3.0,1,349909,21.0750,NaN,S
4,10,1.0,2.0,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1.0,0,237736,30.0708,NaN,C
...,...,...,...,...,...,...,...,...,...,...,...,...
440,882,0.0,3.0,"Markun, Mr. Johann",male,33.0,0.0,0,349257,7.8958,NaN,S
441,884,0.0,2.0,"Banfield, Mr. Frederick James",male,28.0,0.0,0,C.A./SOTON 34068,10.5000,NaN,S
442,886,0.0,3.0,"Rice, Mrs. William (Margaret Norton)",female,39.0,0.0,5,382652,29.1250,NaN,Q
443,888,1.0,1.0,"Graham, Miss. Margaret Edith",female,19.0,0.0,0,112053,30.0000,B42,S


In [4]:
df.shape[0]

445

# 2) Feature Filtering

2.1 Drop variables with more than 50% missing values.

2.2 Check all columns except `Age` and `Fare` for flat values and drop columns where a flat value is more than 70%.

From 2.1 and 2.2, how many columns remain?

Note: Ensure missing values are considered in the calculation. If using `value_counts()`, include `dropna=False`.


In [41]:
# Filter columns by missing and flat-value ratios.
dropped_1 = df.loc[:, df.isnull().mean() <= 0.5]

not_age_and_fare = dropped_1.columns.difference(["Age", "Fare"])
flat_values = dropped_1[not_age_and_fare].apply(
    lambda col: col.value_counts(normalize=True, dropna=False).max()
)
to_drop = flat_values[flat_values > 0.7].index
dropped_2 = dropped_1.drop(columns=to_drop)

dropped_2.shape[1]

11

# 3) Target Cleaning

Remove all rows with missing values in the target variable `Survived`.

How many rows remain?


In [26]:
# Drop rows with missing values in `Survived`.
df[df["Survived"].notnull()].shape[0]

432

# 4) Outlier Handling

For `Fare`, replace outlier values with the boundary values:

- If value < (Q1 - 1.5IQR), replace it with (Q1 - 1.5IQR).
- If value > (Q3 + 1.5IQR), replace it with (Q3 + 1.5IQR).

What is the mean of `Fare` after replacing the outliers, rounded to 2 decimal places?

Hint: Use `round(_, 2)`.


In [32]:
# Calculate the IQR boundaries and replace the outliers

q1 = df["Fare"].quantile(0.25)
q3 = df["Fare"].quantile(0.75)
iqr = q3 - q1

df2 = df

df2[df2["Fare"] < q1 - 1.5 * iqr] = q1 - 1.5 * iqr
df2[df2["Fare"] > q3 + 1.5 * iqr] = q3 + 1.5 * iqr

round(df2["Fare"].mean(), 2)

np.float64(26.27)

# 5) Numeric Imputation

Build a scikit-learn numeric preprocessing pipeline.

Separate the target variable `Survived` from the input features. For numeric input columns, build a scikit-learn `Pipeline` using `SimpleImputer(strategy="mean")`. Use `ColumnTransformer` to apply preprocessing to the selected columns.

What is the mean of `Age` after imputing the missing values, rounded to 2 decimal places?

Hint: Use `Pipeline`, `SimpleImputer`, `ColumnTransformer`, and `round(_, 2)`.


In [161]:
# Separate X and y, then build and fit the numeric preprocessing pipeline.
nums = df.select_dtypes(include="number")

target = ["Survived"]

X_cols = nums.columns.difference(target)
Y_cols = nums.columns.intersection(target)

imp_mean = SimpleImputer(strategy="mean")
ct       = ColumnTransformer([(f"t{i}", imp_mean, [col]) for (i, col) in enumerate(X_cols)])
pipe     = Pipeline([("ct", ct)])

pipe.set_output(transform="pandas")
pipe.fit(df, df)

mean = pipe.transform(df)["t0__Age"].mean()
round(mean, 2)


np.float64(29.14)

In [144]:
nums

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
0,2,1.0,1.0,38.0,1.0,0,71.2833
1,4,1.0,1.0,35.0,1.0,0,53.1000
2,6,0.0,3.0,NaN,0.0,0,8.4583
3,8,0.0,3.0,2.0,3.0,1,21.0750
4,10,1.0,2.0,14.0,1.0,0,30.0708
...,...,...,...,...,...,...,...
440,882,0.0,3.0,33.0,0.0,0,7.8958
441,884,0.0,2.0,28.0,0.0,0,10.5000
442,886,0.0,3.0,39.0,0.0,5,29.1250
443,888,1.0,1.0,19.0,0.0,0,30.0000


# 6) Categorical Encoding

Repeat the numeric preprocessing from Problem 5 and separate the target variable `Survived` from the input features.

For `Embarked`, build a categorical `Pipeline` using `SimpleImputer(strategy="most_frequent")` and `OneHotEncoder`. Use `C` as the reference category, handle unknown categories as all zeros, and return dense output.

Combine the numeric and categorical pipelines using `ColumnTransformer` inside a `Pipeline`.

What is the mean of `Embarked_Q`, rounded to 2 decimal places?

Hint: Check the `drop`, `handle_unknown`, and `sparse_output` options, and use `round(_, 2)`.


In [194]:
# Repeat the numeric preprocessing, add the `Embarked` branch, and fit the combined pipeline.
nums = df.select_dtypes(include="number")

target = ["Survived"]

X_num_cols = nums.columns.difference(target)
X_embarked = df.columns.intersection(["Embarked"])

Y_cols = nums.columns.intersection(target)

ct = ColumnTransformer(
    [
        (
            "t_num",
            SimpleImputer(strategy="mean"),
            X_num_cols
        ),
        (
            "t_cat",
            Pipeline([
                ("imp_freq", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]),
            X_embarked
        ),
        
    ],
    verbose_feature_names_out=False
)

ohe = OneHotEncoder(handle_unknown="ignore")

pipe = Pipeline([("ct", ct)])

pipe.set_output(transform="pandas")
pipe.fit(df, df)

pipe.transform(df)
mean = pipe.transform(df)["Embarked_Q"].mean()
round(mean, 2)

np.float64(0.06)

# 7) Train-Test Splitting

Repeat the target cleaning from Problem 3.

Separate the input features (`X`) and target (`y`). Split the data into training and test sets using a 70:30 ratio, `random_state=123`, and `stratify=y`.

What is the proportion of survivors in the training data, rounded to 2 decimal places?

Hint: Use `dropna()`, `round(_, 2)`, and `train_test_split()` from `sklearn.model_selection`.


In [209]:
# Repeat the target cleaning, separate X and y, and perform the stratified split.
cleaned = df[df["Survived"].notnull()]

X = cleaned.loc[:, cleaned.columns != "Survived"]
y = cleaned["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123, stratify=y)
round(y_train.mean(), 2)

np.float64(0.42)